# 情感陪伴AI - Colab一键训练

运行以下所有单元格，自动完成：
1. 安装依赖
2. 生成训练数据
3. 训练BPE Tokenizer
4. 预训练（学会中文）
5. 情感微调
6. 知识增强微调
7. 导出ONNX模型
8. INT8量化

In [ ]:
# 检查GPU
!nvidia-smi

In [ ]:
# 安装依赖
!pip install torch torchvision torchaudio onnx onnxruntime tokenizers tqdm -q

In [ ]:
# 克隆项目
!git clone https://github.com/stone2010/stgblog.git
%cd stgblog/training_pipeline

In [ ]:
# 生成训练数据
!python generate_pretrain_data.py
!python generate_knowledge_data.py

In [ ]:
# 训练BPE Tokenizer
!python train_tokenizer.py

In [ ]:
# 预训练（学会中文）
!python pretrain.py

In [ ]:
# 情感微调
!python finetune_emotion.py

In [ ]:
# 知识增强微调
!python finetune_knowledge.py

In [ ]:
# 导出ONNX模型
!python export_onnx.py

In [ ]:
# 测试模型
import torch
import json
from tokenizers import Tokenizer
from model_v2 import RomanticLM

tokenizer = Tokenizer.from_file("tokenizer.json")

model = RomanticLM(vocab_size=8000, d_model=256, nhead=4, num_layers=4, 
                  dim_feedforward=1024, max_seq_len=256)
model.load_state_dict(torch.load('model_knowledge.pt', map_location='cpu', weights_only=True))
model.eval()

def generate_text(model, prompt, tokenizer, max_len=50, temperature=0.7):
    input_ids = tokenizer.encode(f"[CLS] {prompt} [SEP]").ids
    input_tensor = torch.tensor([input_ids], dtype=torch.long)
    
    with torch.no_grad():
        for _ in range(max_len):
            logits = model(input_tensor)
            next_token_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            
            if next_token == tokenizer.token_to_id("[END]"):
                break
            
            input_tensor = torch.cat([input_tensor, torch.tensor([[next_token]])], dim=1)
    
    return tokenizer.decode(input_tensor[0].tolist())

test_inputs = ['你好', '今天好累', '想你了', '晚安', '你知道猫吗']
for prompt in test_inputs:
    response = generate_text(model, prompt, tokenizer)
    print(f'{prompt} -> {response}')

In [ ]:
# 下载模型文件
from google.colab import files
files.download('model_knowledge.pt')
files.download('model.onnx')
files.download('model_quantized.onnx')
files.download('tokenizer.json')